<a href="https://colab.research.google.com/github/FatiBuuloloo/BCA_Product_RAG_Conversational_AI-mini_project-010/blob/main/Chatbot_BCA_Product(push_json_%2B_simulation).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain

In [ ]:
pip install langchain-huggingface

In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.0 MB/s eta 0:00:00


In [ ]:
pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
pip install langchain-pinecone

In [ ]:
import os
import json
import requests
import time
from tqdm import tqdm
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import re
from groq import Groq
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings as _HFEmb

In [ ]:
PINECONE_API_KEY = "Pinecone API"
GROQ_API_KEY = "Groq_API"
INDEX_NAME = "chatbot-view-2"
MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 100

# Push JSON to Pinecone

In [ ]:
url = "https://raw.githubusercontent.com/FatiBuuloloo/BCA_Product_RAG_Conversational_AI-mini_project-010/main/Files/bca_produk_individu.json"

response = requests.get(url, timeout=60)
response.raise_for_status()
chunks = response.json()
print(f"Done Total Chunks: {len(chunks)}")

Done Total Chunks: 1052


In [ ]:
model = SentenceTransformer(MODEL_NAME)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("chatbot-view-2")
def sanitize_metadata(meta: dict) -> dict:
    clean = {}
    for k, v in meta.items():
        if isinstance(v, (str, int, float, bool)):
            clean[k] = v
        elif isinstance(v, list):
            clean[k] = [str(i) for i in v]
        else:
            clean[k] = str(v)
    return clean
errors = []
total_upserted = 0
for batch_start in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Batch"):
    batch = chunks[batch_start : batch_start + BATCH_SIZE]
    texts = ["passage: " + item["text"] for item in batch]
    embeddings = model.encode(texts, show_progress_bar=False).tolist()
    vectors = []
    for item, embedding in zip(batch, embeddings):
        try:
            meta = sanitize_metadata(item["metadata"])
            meta["text"] = item["text"]
            vectors.append({
                "id"     : item["id"],
                "values" : embedding,
                "metadata": meta
            })
        except Exception as e:
            errors.append({"id": item.get("id"), "error": str(e)})
    if vectors:
        index.upsert(vectors=vectors)
        total_upserted += len(vectors)
    time.sleep(0.2)

In [ ]:
info = pc.describe_index("chatbot-view-2")
print("Dimensi index:", info.dimension)
print("Status:", info.status)

Dimensi index: 1024
Status: {'ready': True, 'state': 'Ready'}


# Simulation and Recheck

In [ ]:
class BGEEmbeddings(_HFEmb):
    def embed_query(self, text: str):
        return super().embed_query("query: " + text)

lc_embeddings = BGEEmbeddings(model_name=MODEL_NAME,encode_kwargs={"normalize_embeddings": True})

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=lc_embeddings,
    pinecone_api_key=PINECONE_API_KEY
)
client = Groq(api_key=GROQ_API_KEY)
MAX_CONTEXT_CHARS = 6000

In [ ]:
products = [
    # Simpanan Individu
    "Tahapan BCA", "Tahapan Xpresi",
    "Tahapan Berjangka", "Tahapan Berjangka SiMuda",
    "Simpanan Pelajar", "Tabungan Prestasi (Tapres)", "TabunganKu",
    "Deposito Berjangka", "BCA Dollar", "e-Deposito",
    # Pinjaman Individu
    "Kredit Sepeda Motor (KSM)",
    "Kredit Kendaraan Bermotor Pembelian",
    "Kredit Kendaraan Bermotor Refinancing",
    "Kredit Pemakaian Rumah Pembelian",
    "Kredit Pemakaian Rumah (KPR) Pembelian",
    "Kredit Pemakaian Rumah Sejahtera BCA",
    "Kredit Pemakaian Rumah Refinancing",
    "Kredit Pemakaian Rumah Renovasi",
    "Pinjaman Kredit Tanpa Agunan Personal",
    "BCA Secured Personal Loan",
    # Wealth Management — Asuransi
    "Asuransi Kebakaran", "Asuransi Property All Risks",
    "Asuransi Electronic Equipment Insurance (EEI)",
    "Asuransi MyGuard",
    "Asuransi MyGuard - Accident Care",
    "Asuransi MyGuard - Critical Care",
    "Asuransi MyGuard - Health Care",
    "Asuransi MyGuard - Hospital Care",
    "Asuransi BCA Life Proteksi Jiwa Optima",
    "Asuransi Maxi Infinite Link Assurance Plus (MILA Plus)",
    "Asuransi Proteksi Jiwa Maksima (JIMI)",
    "Asuransi Credit Life", "Asuransi Credit Life Chubb Life",
    "Asuransi Credit Life Paylater",
    "Asuransi Heritage Platinum Protection (Heritage+)",
    "Asuransi BCA Life Accident Safeguard",
    "Asuransi Optima Accident Protection",
    "Asuransi Education Guard", "Asuransi Household Guard",
    "Asuransi Kendaraan Bermotor",
    "Asuransi Safety Guard Critical Cover (STAR)",
    "Asuransi AIA Optima Protection Plus",
    "Asuransi BCA Life Perlindungan Kritis Optima",
    "Asuransi Bima Proteksi Kesehatanku",
    "Asuransi Dental Care Plan",
    "Asuransi Hospital 100% Refundable",
    "Asuransi Maxi Value Protection",
    "Asuransi Optima Cancer Protection",
    "Asuransi Proteksi Kesehatan Ultima",
    "Asuransi Proteksi Penyakit Kritis Maksima Extra (Prima Extra)",
    "Asuransi Smart Eazicare Blue",
    "Asuransi Smart Eazicare Platinum",
    "Asuransi Proteksi Edukasi Maksima",
    "Asuransi Proteksi Retirement Maksima",
    "Asuransi Proteksi Wholelife Income Maksima",
    "Travel Insurance",
    "Asuransi Prosper Life Guard (PROSPER)",
    # Wealth Management — Investasi
    "BCA Reksadana", "Obligasi",
    "Rekening Dana Nasabah", "Rekening Dana Lender (RDL) BCA",
    # Kartu Kredit BCA
    "BCA Everyday Card", "BCA Card Platinum", "BCA Smartcash",
    "BCA Singapore Airlines KrisFlyer Visa Signature",
    "BCA Singapore Airlines KrisFlyer Visa Infinite",
    "BCA Singapore Airlines PPS Club Visa Infinite",
    "BCA Mastercard Black", "BCA Blibli Mastercard",
    "BCA tiket.com Mastercard",
    "BCA Mastercard Globe", "BCA Mastercard World",
    "BCA JCB Black", "BCA UnionPay", "BCA American Express Platinum",
    # Uang Elektronik
    "Flazz", "Sakuku",
    # Reward
    "Reward BCA",
]

product_lists = {
    "Simpanan Individu": [
        "Tahapan BCA", "Tahapan Xpresi",
        "Tahapan Berjangka", "Tahapan Berjangka SiMuda",
        "Simpanan Pelajar", "Tabungan Prestasi (Tapres)", "TabunganKu",
        "Deposito Berjangka", "BCA Dollar", "e-Deposito",
    ],
    "Pinjaman Individu": [
        "Kredit Sepeda Motor (KSM)",
        "Kredit Kendaraan Bermotor Pembelian",
        "Kredit Kendaraan Bermotor Refinancing",
        "Kredit Pemakaian Rumah Pembelian",
        "Kredit Pemakaian Rumah Sejahtera BCA",
        "Kredit Pemakaian Rumah Refinancing",
        "Kredit Pemakaian Rumah Renovasi",
        "Pinjaman Kredit Tanpa Agunan Personal",
        "BCA Secured Personal Loan",
    ],
    "Uang Elektronik": ["Flazz", "Sakuku"],
    "Kartu Kredit BCA": [
        "BCA Everyday Card", "BCA Card Platinum", "BCA Smartcash",
        "BCA Singapore Airlines KrisFlyer Visa Signature",
        "BCA Singapore Airlines KrisFlyer Visa Infinite",
        "BCA Singapore Airlines PPS Club Visa Infinite",
        "BCA Mastercard Black", "BCA Blibli Mastercard",
        "BCA tiket.com Mastercard", "BCA Mastercard Globe",
        "BCA Mastercard World", "BCA JCB Black",
        "BCA UnionPay", "BCA American Express Platinum",
    ],
    "Reward BCA": ["Reward BCA"],
}

wealth_lists = """**Wealth Management BCA** terdiri dari:

**Asuransi** (berdasarkan jenis):
- Harta Benda: Asuransi Kebakaran, Property All Risks, Electronic Equipment Insurance (EEI)
- Jiwa: MyGuard (Accident/Critical/Health/Hospital Care), BCA Life Proteksi Jiwa Optima, MILA Plus, JIMI, Credit Life, Credit Life Chubb Life, Credit Life Paylater, Heritage Platinum Protection
- Kecelakaan: BCA Life Accident Safeguard, Optima Accident Protection, Education Guard, Household Guard
- Kendaraan: Asuransi Kendaraan Bermotor
- Kesehatan: Safety Guard Critical Cover (STAR), AIA Optima Protection Plus, BCA Life Perlindungan Kritis Optima, Bima Proteksi Kesehatanku, Dental Care Plan, Hospital 100% Refundable, Maxi Value Protection, Optima Cancer Protection, Proteksi Kesehatan Ultima, Prima Extra, Smart Eazicare Blue, Smart Eazicare Platinum
- Pendidikan: Proteksi Edukasi Maksima
- Pensiun & Anuitas: Proteksi Retirement Maksima, Proteksi Wholelife Income Maksima
- Travel: Travel Insurance
- Warisan: Heritage Platinum Protection (Heritage+), Prosper Life Guard (PROSPER)

**Investasi**: BCA Reksadana, Obligasi

**Rekening Investasi**: Rekening Dana Nasabah (RDN), Rekening Dana Lender (RDL) BCA"""

product_prompt = "\n".join(f"- {p}" for p in products)
analyze_prompt = f"""Anda adalah sistem pendeteksi nama produk BCA dari pertanyaan user.

TUGAS:
1. product_name          : nama produk PERSIS dari daftar di bawah, atau null
2. needs_clarification   : true jika ambigu (tidak menyebut produk spesifik)
3. clarification_question: kalimat tanya sopan Bahasa Indonesia
4. clarification_options : array pilihan yang relevan

DAFTAR PRODUK (salin PERSIS termasuk tanda baca, spasi, dan huruf kapital):
{product_prompt}

PEMETAAN PENTING — jika user menyebut ini, gunakan nama di sebelah kanan:
- "kpr pembelian" / "kpr bca pembelian"           → "Kredit Pemakaian Rumah Pembelian"
- "kpr sejahtera" / "kpr subsidi"                 → "Kredit Pemakaian Rumah Sejahtera BCA"
- "kpr refinancing"                               → "Kredit Pemakaian Rumah Refinancing"
- "kpr renovasi"                                  → "Kredit Pemakaian Rumah Renovasi"
- "pinjaman tanpa agunan" / "kta" / "personal loan" (bukan secured) → "Pinjaman Kredit Tanpa Agunan Personal"
- "secured personal loan" / "pinjaman agunan investasi" → "BCA Secured Personal Loan"
- "myguard critical" / "critical care"            → "Asuransi MyGuard - Critical Care"
- "myguard accident"                              → "Asuransi MyGuard - Accident Care"
- "myguard health"                                → "Asuransi MyGuard - Health Care"
- "myguard hospital"                              → "Asuransi MyGuard - Hospital Care"
- "optima cancer" / "asuransi kanker"             → "Asuransi Optima Cancer Protection"
- "aia optima" / "optima protection plus"         → "Asuransi AIA Optima Protection Plus"
- "tapres"                                        → "Tabungan Prestasi (Tapres)"
- "jimi"                                          → "Asuransi Proteksi Jiwa Maksima (JIMI)"
- "mila plus" / "mila"                            → "Asuransi Maxi Infinite Link Assurance Plus (MILA Plus)"
- "star" / "safety guard"                         → "Asuransi Safety Guard Critical Cover (STAR)"
- "hoki"                                          → "Asuransi Proteksi Wholelife Income Maksima"
- "prosper"                                       → "Asuransi Prosper Life Guard (PROSPER)"
- "pratama"                                       → "Asuransi Proteksi Kesehatan Ultima"

needs_clarification=true HANYA jika:
- User tanya "KPR" tanpa jenis (pembelian/sejahtera/refinancing/renovasi)
- User tanya "pinjaman"/"kredit" tanpa produk spesifik
- User tanya "asuransi" tanpa nama produk spesifik
- User tanya dokumen/persyaratan suatu produk TANPA menyebut jenis nasabah (karyawan/wiraswasta/profesional) — dalam kasus ini, tanyakan jenis nasabahnya dan berikan pilihan: ["Karyawan", "Wiraswasta", "Profesional"]

PENTING: Kembalikan HANYA satu baris JSON. DILARANG menambahkan penjelasan atau markdown. Output:
{{"product_name": "...", "needs_clarification": false, "clarification_question": "", "clarification_options": []}}"""

answer_prompt = """\
Anda adalah "View", asisten virtual resmi BCA.
Pengembang: Fati Buulolo | LinkedIn: https://www.linkedin.com/in/fati-buulolo-7a9236391/

KONTEKS — satu-satunya sumber kebenaran:
{context}

ATURAN MUTLAK:
1. Jawab HANYA dari teks yang ada di KONTEKS.
2. DILARANG menambahkan, mengarang, atau menyimpulkan dari pengetahuan umum.
3. Tampilkan SEMUA angka, poin, dan detail yang ada di KONTEKS — jangan diringkas.
4. TAMPILKAN SEMUA informasi dari KONTEKS yang relevan dengan pertanyaan user — termasuk sub-poin, angka, dan kondisi. JANGAN lewati atau singkat bagian apapun.
5. Jika info yang ditanya memang tidak ada di KONTEKS, jawab:
   "Maaf, informasi tersebut tidak tersedia dalam data saya. Silakan hubungi Halo BCA di **1500 888**."
6. Gunakan nama produk PERSIS seperti di KONTEKS.
7. Format: bullet points, **bold** untuk angka dan istilah penting, URL sebagai teks biasa.
7. Nilai uang: format Rupiah (Rp50.000).
8. Tutup dengan: Info lebih lanjut: Halo BCA **1500 888**.
"""


In [ ]:
def is_list_query(question: str):
    q = question.lower()
    trigger_keywords = [
        "list produk", "daftar produk", "sebutkan produk",
        "apa saja produk", "semua produk", "jenis produk",
        "tampilkan produk", "produk apa saja", "produk-produk",
    ]
    specific_key_words = [
        "persyaratan", "syarat", "dokumen", "biaya", "bunga",
        "suku bunga", "limit", "plafon", "manfaat", "fitur",
        "risiko", "cara", "berapa", "jelaskan", "premi",
        "denda", "pengecualian", "ketentuan",
    ]
    if not any(kw in q for kw in trigger_keywords):
        return False, None
    if any(kw in q for kw in specific_key_words):
        return False, None
    if any(w in q for w in ["wealth","asuransi","investasi","reksadana","rdn","rdl"]):
        return True, "Wealth Management"
    if any(w in q for w in ["simpanan","tabungan","deposito","giro"]):
        return True, "Simpanan Individu"
    if any(w in q for w in ["pinjaman individu","kredit individu","jenis pinjaman"]):
        return True, "Pinjaman Individu"
    if "kartu kredit" in q:
        return True, "Kartu Kredit BCA"
    if any(w in q for w in ["uang elektronik"]):
        return True, "Uang Elektronik"
    if "reward" in q:
        return True, "Reward BCA"
    return False, None


def answer_list(category: str) -> str:
    if category == "Wealth Management":
        return "Berikut produk **Wealth Management BCA**:\n\n" + wealth_lists + \
               "\n\nInfo lebih lanjut: Halo BCA **1500 888**."
    items = "\n".join(f"- **{p}**" for p in product_lists.get(category, []))
    return f"Berikut seluruh produk **{category}**:\n\n{items}\n\nInfo lebih lanjut: Halo BCA **1500 888**."

def analyze_intent(question: str, verbose=False) -> dict:
    resp = client.chat.completions.create(
        messages=[
            {"role": "system", "content": analyze_prompt},
            {"role": "user",   "content": f"Pertanyaan: {question}"}
        ],
        model="llama-3.1-8b-instant",
        temperature=0.0,
        max_tokens=150,
    )
    raw = resp.choices[0].message.content.strip()
    if verbose:
        print(f"Analyze → {raw}")
    try:
        return json.loads(raw)
    except:
        m = re.search(r'\{.*?\}', raw, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except: pass
    return {"product_name": None, "needs_clarification": False,
            "clarification_question": "", "clarification_options": []}

def normalize(s: str) -> str:
    return re.sub(r'[\s\-–—()/.,]+', ' ', s.strip().lower())

def retrieve(question: str, product_name: str | None, verbose=False) -> str:
    docs = vectorstore.similarity_search(question, k=20)
    if not docs:
        return ""
    if product_name:
        pn_norm = normalize(product_name)
        target  = [d for d in docs if normalize(d.metadata.get("product_name","")) == pn_norm]
        final   = target if target else docs
        if verbose:
            print(f"Looking Chunk of '{product_name}': {len(target)} (total semantic: {len(docs)})")
    else:
        final = docs
        if verbose:
            top = [d.metadata.get('product_name','?') for d in docs[:3]]
            print(f" Looking for Top-3 semantic (tanpa filter): {top}")
    parts, total, seen = [], 0, set()
    for i, doc in enumerate(final, 1):
        meta   = doc.metadata
        sec_id = meta.get("section_id", f"_no_id_{i}")
        if sec_id in seen:
            continue
        seen.add(sec_id)
        body = meta.get("text") or doc.page_content
        header = (
            f"[Chunk {i}]"
            f" Produk: {meta.get('product_name','-')}"
            f" | Bagian: {meta.get('section_title','-')}"
            f" | Tag: {meta.get('topic_tag','-')}"
        )
        chunk = f"{header}\n{body}"
        if total + len(chunk) > MAX_CONTEXT_CHARS:
            sisa = MAX_CONTEXT_CHARS - total
            if sisa > 200:
                parts.append(chunk[:sisa] + "\n[...dipotong]")
            break
        parts.append(chunk)
        total += len(chunk)
    return "\n\n---\n\n".join(parts)


def generate_answer(question: str, context: str, history: list) -> str:
    recent   = history[-4:] if len(history) > 4 else history
    messages = [{"role": "system", "content": answer_prompt.format(context=context)}]
    messages.extend(recent)
    messages.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        messages=messages,
        model="llama-3.1-8b-instant",
        temperature=0.0,
        max_tokens=900,
        top_p=0.9,
    )
    return resp.choices[0].message.content

def ask_view(question: str, history: list = None, verbose: bool = False):
    if history is None:
        history = []
    q_lower = question.lower()
    if any(w in q_lower for w in ["pengembang","siapa yang buat","siapa pembuatmu",
                                   "siapa penciptamu","dibuat oleh","dikembangkan oleh",
                                   "siapakah kamu","siapa kamu","who made","developer"]):
        return (
            "Saya adalah **View**, asisten virtual untuk informasi produk BCA.\n\n"
            "Saya dikembangkan oleh **Fati Buulolo**, seorang mahasiswa Jurusan Matematika "
            "dengan fokus data dan Machine Learning.\n"
            "LinkedIn: https://www.linkedin.com/in/fati-buulolo-7a9236391/\n\n"
            "Info lebih lanjut tentang produk BCA: Halo BCA **1500 888**."
        ), None, {}
    q_lower = question.lower()
    if any(w in q_lower for w in ["pengembang", "siapa yang membuat", "siapa yang mengembangkan",
                                   "dibuat oleh", "developer", "who made", "siapa kamu",
                                   "siapa pembuat", "tentang view", "about view"]):
        return (
            "Saya **View** 🏦, asisten virtual untuk informasi produk BCA.\n\n"
            "Saya dikembangkan oleh **Fati Buulolo**, mahasiswa Jurusan Matematika "
            "dengan fokus Machine Learning.\n\n"
            "LinkedIn: https://www.linkedin.com/in/fati-buulolo-7a9236391/\n\n"
            "Info lebih lanjut tentang produk BCA: Halo BCA **1500 888**."
        ), None, {}
    is_list, list_cat = is_list_query(question)
    if is_list and list_cat:
        return answer_list(list_cat), None, {}
    intent = analyze_intent(question, verbose=verbose)
    if verbose:
        print(f" product_name : {intent.get('product_name')}")
        print(f" clarify      : {intent.get('needs_clarification')}")
    if intent.get("needs_clarification") and intent.get("clarification_question"):
        clarif_q   = intent["clarification_question"]
        clarif_opt = intent.get("clarification_options", [])
        clarif_txt = clarif_q
        if clarif_opt:
            opts      = "\n".join(f"  {i+1}. {o}" for i, o in enumerate(clarif_opt))
            clarif_txt = f"{clarif_q}\n\n{opts}"
        return clarif_txt, "NEEDS_CLARIFICATION", intent
    context = retrieve(question, intent.get("product_name"), verbose=verbose)
    if not context.strip():
        return ("Maaf, informasi tersebut tidak tersedia dalam data saya. "
                "Silakan hubungi Halo BCA di **1500 888**."), None, {}
    answer = generate_answer(question, context, history)
    return answer, None, intent

exit_keywords = (
    "keluar", "exit", "quit", "selesai", "stop",
    "terima kasih", "makasih", "thanks", "thank you",
    "bye", "sampai jumpa", "sudah selesai", "sudah cukup",
    "tidak ada lagi", "tidak ada pertanyaan",
    "selesai bertanya", "saya selesai", "saya sudah", "cukup",
)

def chat(verbose: bool = False):
    history = []
    pending = {}

    GREETING = """
Halo! Selamat datang di **View**, asisten virtual untuk
informasi produk BCA. Saya siap membantu Anda :)!

Berikut beberapa hal yang bisa Anda tanyakan:
  - Apa saja produk simpanan individu BCA?
  - Berapa suku bunga promo KPR BCA Pembelian?
  - Apa persyaratan mengajukan pinjaman tanpa agunan BCA?
  - Jelaskan manfaat Asuransi MyGuard Critical Care.
  - Apa saja daftar premi Asuransi AIA Optima Protection Plus?
  - Siapakah yang mengembangkan kamu?

Ketik 'keluar' untuk mengakhiri sesi."""
    print(GREETING)
    while True:
        try:
            user_input = input("\n🧑 Anda : ").strip()
        except KeyboardInterrupt:
            print("\n\n🤖 View  : Sesi dihentikan. Terima kasih telah menggunakan View! "
                  "Hubungi Halo BCA **1500 888** jika ada pertanyaan. Sampai jumpa! 👋")
            break
        except EOFError:
            print("\n🤖 View  : Sesi berakhir. Sampai jumpa! 👋")
            break
        if not user_input:
            continue
        if any(kw in user_input.lower() for kw in exit_keywords):
            print("\n🤖 View  : Terima kasih sudah menggunakan View! Semoga informasi "
                  "yang diberikan bermanfaat. Sampai jumpa! 👋")
            break
        if pending.get("waiting_clarification"):
            original_q   = pending["original_question"]
            saved_intent = pending.get("intent", {})
            combined_q   = f"{original_q} {user_input}"
            print("\n🤖 View  :", end=" ", flush=True)
            context = retrieve(combined_q, saved_intent.get("product_name"), verbose=verbose)
            if not context.strip():
                answer = ("Maaf, informasi tersebut tidak tersedia. "
                          "Hubungi Halo BCA **1500 888**.")
            else:
                answer = generate_answer(combined_q, context, history)
            print(answer)
            history.append({"role": "user",      "content": combined_q})
            history.append({"role": "assistant",  "content": answer})
            if len(history) > 8:
                history = history[-8:]
            pending = {}
            continue
        print("\n🤖 View  :", end=" ", flush=True)
        answer, flag, _intent = ask_view(user_input, history=history, verbose=verbose)
        print(answer)
        if flag == "NEEDS_CLARIFICATION":
            pending = {
                "waiting_clarification": True,
                "original_question":     user_input,
                "intent":                _intent,
            }
        else:
            history.append({"role": "user",      "content": user_input})
            history.append({"role": "assistant",  "content": answer})
            if len(history) > 8:
                history = history[-8:]

In [ ]:
chat(verbose=True)


Halo! Selamat datang di **View**, asisten virtual untuk
informasi produk BCA. Saya siap membantu Anda :)! 

Berikut beberapa hal yang bisa Anda tanyakan:
  - Apa saja produk simpanan individu BCA?
  - Berapa suku bunga promo KPR BCA Pembelian?
  - Apa persyaratan mengajukan pinjaman tanpa agunan BCA?
  - Jelaskan manfaat Asuransi MyGuard Critical Care.
  - Apa saja daftar premi Asuransi AIA Optima Protection Plus?
  - Siapakah yang mengembangkan kamu?

Ketik 'keluar' untuk mengakhiri sesi.

🧑 Anda : Apa saja produk simpanan individu BCA?

🤖 View  : Berikut seluruh produk **Simpanan Individu**:

- **Tahapan BCA**
- **Tahapan Xpresi**
- **Tahapan Berjangka**
- **Tahapan Berjangka SiMuda**
- **Simpanan Pelajar**
- **Tabungan Prestasi (Tapres)**
- **TabunganKu**
- **Deposito Berjangka**
- **BCA Dollar**
- **e-Deposito**

Info lebih lanjut: Halo BCA **1500 888**.

🧑 Anda : Berapa suku bunga promo KPR BCA Pembelian?

🤖 View  : Analyze → {"product_name": "Kredit Pemakaian Rumah Pembelian", "

In [ ]:
chat(verbose=False)


Halo! Selamat datang di **View**, asisten virtual untuk
informasi produk BCA. Saya siap membantu Anda :)! 

Berikut beberapa hal yang bisa Anda tanyakan:
  - Apa saja produk simpanan individu BCA?
  - Berapa suku bunga promo KPR BCA Pembelian?
  - Apa persyaratan mengajukan pinjaman tanpa agunan BCA?
  - Jelaskan manfaat Asuransi MyGuard Critical Care.
  - Apa saja daftar premi Asuransi AIA Optima Protection Plus?
  - Siapakah yang mengembangkan kamu?

Ketik 'keluar' untuk mengakhiri sesi.

🧑 Anda : Jelaskan manfaat Asuransi MyGuard Critical Care.

🤖 View  : Manfaat Asuransi MyGuard – Critical Care mencakup pilihan plan, manfaat meninggal dunia, dan asuransi tambahan Critical Care.

Produk: Asuransi MyGuard - Critical Care | Bagian: Manfaat | Tag: manfaat
Produk: Asuransi MyGuard - Critical Care | Kategori: Wealth Management | Bagian: Manfaat

Info lebih lanjut: Halo BCA **1500 888**.

🧑 Anda : bisakah kamu berikan manfaat meninggal dunianya ?

🤖 View  : Manfaat Meninggal Dunia Asuran

# Remove all vector in pinecone

In [ ]:
import pinecone
from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("chatbot-view-2")
index.delete(delete_all=True)

{}